In [ ]:
prefix_path = '../..'

import sys
sys.path.append(prefix_path)

import torch
import os
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

from egh_vlm.extract_output import batch_extract_output
from egh_vlm.utils import ModelBundle, load_phd_dataset

## Analysis

In [ ]:
dataset_path = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
img_folder_path = f'{prefix_path}/data/phd/images'
output_masked_all_path = f'{prefix_path}/data/phd/output_masked_all.pt'
output_masked_image_path = f'{prefix_path}/data/phd/output_masked_image.pt'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Qwen3VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    dtype='auto',
    device_map=device
)
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    max_pixels=1024 * 768)
model_bundle = ModelBundle(model, processor, device)

In [ ]:
dataset = load_phd_dataset(
    dataset_path=dataset_path,
    img_folder_path=img_folder_path,
    sample_size=500,
)

output_masked_all = torch.load(output_masked_all_path) if os.path.isfile(output_masked_all_path) else {}
print(f'Length of processed output_masked_all: {len(output_masked_all)}')
output_masked_image = torch.load(output_masked_image_path) if os.path.isfile(output_masked_image_path) else {}
print(f'Length of processedoutput_masked_image: {len(output_masked_image)}')

In [ ]:
output_masked_all = batch_extract_output(
    dataset,
    model_bundle,
    processed_outputs=output_masked_all,
    mask_mode='all',
    save_path=output_masked_all_path,
)

In [ ]:
output_masked_image = batch_extract_output(
    dataset,
    model_bundle,
    processed_outputs=output_masked_image,
    mask_mode='image',
    save_path=output_masked_image_path,
)